# FLUX.1-Kontext-dev: сезонная генерация (self-contained)

Код и конфиги зашиты в ячейки, данные LEVIR скачиваются с HF.
Порядок: выполнить ячейки сверху вниз; после smoke глазами
проверить результат, затем полный прогон.

Runtime -> Change runtime type -> **A100** (или L4).

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU не выдан'
gpu = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
if vram_gb >= 35:
    DTYPE, OFFLOAD = 'bf16', 'none'
elif vram_gb >= 20:
    DTYPE, OFFLOAD = 'fp8', 'model'
else:
    # T4: fp8/offload умирает по RAM (12.7GB); проверенный путь -
    # GGUF Q4 + T5-8bit целиком в VRAM (см. model_flux gguf-q4)
    DTYPE, OFFLOAD = 'gguf-q4', 'none'
print(f'{gpu} | {vram_gb:.0f} GB -> dtype={DTYPE}, offload={OFFLOAD}')

In [ ]:
import pathlib
for d in ['src', 'prompts', 'data/input', 'outputs/generated']:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print('структура создана')

In [ ]:
%%writefile config.yaml
# Все параметры генерации фиксированы одинаковыми для всех промптов —
# иначе сравнение типов промптов нечестное.

generation:
  backend: mock                 # mock | flux | flux2 | sdxl | ip2p  (mock = без GPU)
  model_id: black-forest-labs/FLUX.1-Kontext-dev
  dtype: bf16                   # bf16 полная точность | fp8 (24GB) | gguf-q4 (T4 16GB)
  gguf_url: https://huggingface.co/QuantStack/FLUX.1-Kontext-dev-GGUF/blob/main/flux1-kontext-dev-Q4_K_S.gguf
  resolution: 1024              # LEVIR-тайлы 1024x1024
  num_inference_steps: 28
  guidance_scale: 2.5           # рекомендованное для Kontext
  true_cfg_scale: 3.0           # включает реальный negative_prompt; ТОЛЬКО тип negative
  seed: 42
  num_variants: 1               # N изображений на комбинацию: seed, seed+1, ...
  cpu_offload: none             # none (весь пайплайн в VRAM) | model | sequential

dataset:
  input_dir: data/input
  images: []                    # пусто = все файлы из input_dir; иначе список имён

seasons: [summer, autumn, winter, deep_winter]
prompt_types: [zero_shot, contextual, positive, negative, few_shot, cot,
               step_back, decomposed, kontext_guide, hybrid]

evaluation:
  structure_model: timm/vit_small_plus_patch16_dinov3.lvd1689m  # DINOv3, не gated
  season_model: openai/clip-vit-base-patch16
  device: cpu                   # cpu | cuda
  source_season: summer         # исходный сезон снимков, для CLIP directional
  # Порог откалиброван по визуальной сверке FLUX-прогона 20260706-131826:
  # сломанные сцены (другая местность/ракурс) имеют dino < 0.3,
  # живые > 0.4; SSIM у генеративных редакторов систематически низок
  # (перерисовка текстур), поэтому его вес в структуре понижен
  structure_threshold: 0.30
  structure_weights:
    dino: 0.8
    ssim: 0.2
  weights:
    structure: 0.6              # для change detection структура важнее сезона
    season: 0.4

output:
  generated_dir: outputs/generated
  evaluation_dir: outputs/evaluation


In [ ]:
%%writefile prompts/prompts.yaml
# 10 типов x 4 сезона. Тексты на английском — T5-энкодер FLUX его понимает лучше.
# base-часть одинакова у всех типов, меняется только сезонный блок: иначе
# сравнение типов нечестное. Формулировки по гайду BFL для Kontext:
# change/replace вместо transform, явная фиксация placement/camera/framing.
# Плейсхолдеры из src/prompts.py: {season_name} {season_signals} {preserve} {avoid} {scene}

base:
  scene: "high-resolution aerial/satellite top-down photo of an urban area"
  preserve: >-
    keep every road, building, parking lot and field boundary in the exact
    same position and scale; maintain identical layout, camera angle,
    framing and perspective
  avoid: >-
    new buildings, new roads, removed or moved structures, changed street
    layout, different location, distortion, warping, added vehicles or
    objects, blur, artifacts, text, watermark

seasons:
  summer:
    season_name: "summer"
    season_signals: "lush green vegetation, green trees and grass, dry bare ground, warm daylight, no snow"
  autumn:
    season_name: "autumn"
    season_signals: "yellow and orange foliage, fading vegetation, golden and brown tones, some bare trees, no snow"
  winter:
    season_name: "winter"
    # v4 (по эксперименту W2g35): лексика "first snowfall" + запрет заливки;
    # для зимы guidance 3.5 даёт структуру +0.11 и сезон +0.06 против v3
    season_signals: "very light first snowfall: thin white patches scattered between bare dark vegetation and soil, most of the ground and roads still snow-free and clearly visible, pale cold light; do not cover the entire ground with snow"
  deep_winter:
    season_name: "deep winter"
    season_signals: "heavy thick snow covering the entire ground and rooftops, frozen surfaces, strong cold white palette, snow-laden bare trees"

# has_negative: true -> генерация с negative_prompt (true_cfg у FLUX,
# нативный negative у SDXL/IP2P)
prompt_types:

  zero_shot:
    has_negative: false
    template: "Make this a {season_name} scene."

  contextual:
    has_negative: false
    template: >-
      This is a {scene}. Change only the season to {season_name}
      ({season_signals}). Preserve the structure of the territory:
      {preserve}. Do not add, remove or move any structures.

  positive:
    has_negative: false
    template: >-
      Change the season of this {scene} to {season_name}: {season_signals}.
      Realistic remote-sensing look with consistent natural lighting and
      colors typical for {season_name}. {preserve}.

  negative:
    has_negative: true
    template: >-
      Change the season of this {scene} to {season_name}: {season_signals}.
      Realistic remote-sensing look. {preserve}.
    negative_template: "{avoid}"

  few_shot:
    has_negative: false
    template: >-
      Season-editing examples - "summer: green vegetation, dry ground, no
      snow, same roads and buildings"; "autumn: golden foliage, same layout
      and structures". Now apply the same kind of edit: "{season_name}:
      {season_signals}; {preserve}".

  cot:
    has_negative: false
    template: >-
      Step 1: identify all roads, buildings, parking lots and field
      boundaries and keep them exactly in place. Step 2: change the season
      to {season_name} by adding {season_signals}. Step 3: do not add or
      remove any structures and do not change the layout. Final result:
      the same territory shown in {season_name}.

  kontext_guide:
    has_negative: false
    template: >-
      Change the season to {season_name}: {season_signals}. Only replace
      the environmental conditions around the structures. Maintain
      identical placement of every road, building, parking lot and field
      boundary, and keep the same camera angle, framing and perspective.
      Do not add or remove any objects.

  step_back:
    has_negative: false
    template: >-
      What distinguishes a {season_name} aerial photo from a summer one?
      Only the seasonal state of surfaces and vegetation: {season_signals}.
      The territory layout never changes with seasons. Now apply exactly
      this seasonal change to this {scene}: {preserve}.

  decomposed:
    has_negative: false
    template: >-
      Task: seasonal edit of a {scene}. Season: {season_name}
      ({season_signals}). Keep: {preserve}. Forbid: adding, removing or
      moving any structures. Result check: the same territory from the
      same viewpoint, only the seasonal appearance has changed.

  hybrid:
    # запреты внутри промпта, не через true_cfg-negative: у FLUX он
    # статистически вредил структуре (p<0.001)
    has_negative: false
    template: >-
      This is a {scene}. Change the season to {season_name}:
      {season_signals}. Only replace the environmental conditions around
      the structures; realistic remote-sensing look with natural
      {season_name} lighting. Maintain identical placement of every road,
      building, parking lot and field boundary, and keep the same camera
      angle, framing and perspective. Do not add new buildings, roads or
      vehicles, do not remove or move any structures, avoid distortion
      of the layout.


In [ ]:
%%writefile src/prompts.py
"""Сборка матрицы промптов (сезон x тип) из prompts.yaml + config.yaml."""
from __future__ import annotations

import re
from dataclasses import dataclass
from pathlib import Path

import yaml

ROOT = Path(__file__).resolve().parent.parent


@dataclass
class PromptItem:
    season: str
    ptype: str
    prompt: str
    negative_prompt: str | None


def _load_yaml(path: Path) -> dict:
    with path.open(encoding="utf-8") as f:
        return yaml.safe_load(f)


def load_config(path: Path | None = None) -> dict:
    return _load_yaml(path or ROOT / "config.yaml")


def load_prompts(path: Path | None = None) -> dict:
    return _load_yaml(path or ROOT / "prompts" / "prompts.yaml")


def _fmt(template: str, ctx: dict) -> str:
    text = template.format(**ctx)
    return re.sub(r"\s+", " ", text).strip()


def build_prompt_matrix(config: dict, prompts: dict) -> list[PromptItem]:
    base = prompts["base"]
    seasons = prompts["seasons"]
    ptypes = prompts["prompt_types"]

    items: list[PromptItem] = []
    for season_key in config["seasons"]:
        season = seasons[season_key]
        ctx = {**base, **season}
        for ptype_key in config["prompt_types"]:
            spec = ptypes[ptype_key]
            prompt = _fmt(spec["template"], ctx)
            negative = None
            if spec.get("has_negative"):
                negative = _fmt(spec["negative_template"], ctx)
            items.append(PromptItem(season_key, ptype_key, prompt, negative))
    return items


def main() -> None:
    config = load_config()
    prompts = load_prompts()
    matrix = build_prompt_matrix(config, prompts)
    print(f"Матрица: {len(matrix)} промптов "
          f"({len(config['seasons'])} сезонов x {len(config['prompt_types'])} типов)\n")
    for it in matrix:
        print(f"[{it.season} / {it.ptype}]")
        print(f"  prompt:   {it.prompt}")
        if it.negative_prompt:
            print(f"  negative: {it.negative_prompt}")
        print()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/gdal_io.py
"""Чтение/запись снимков в RGB uint8 (H, W, 3)."""
from __future__ import annotations

from pathlib import Path

import numpy as np

try:
    from osgeo import gdal  # type: ignore
    gdal.UseExceptions()
    HAS_GDAL = True
except ImportError:
    HAS_GDAL = False

SUPPORTED_EXT = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}


def _to_uint8(band: np.ndarray) -> np.ndarray:
    if band.dtype == np.uint8:
        return band
    band = band.astype(np.float64)
    lo, hi = float(band.min()), float(band.max())
    if hi <= lo:
        return np.zeros(band.shape, dtype=np.uint8)
    return (((band - lo) / (hi - lo)) * 255.0).round().astype(np.uint8)


def _read_gdal(path: Path) -> tuple[np.ndarray, dict]:
    ds = gdal.Open(str(path))
    arr = ds.ReadAsArray()  # GDAL отдаёт band-first (bands, H, W), не H, W, C
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=0)
    if arr.shape[0] > 3:
        arr = arr[:3]
    elif arr.shape[0] == 2:
        arr = np.concatenate([arr, arr[:1]], axis=0)
    rgb = np.stack([_to_uint8(b) for b in arr], axis=-1)
    meta = {
        "reader": "gdal",
        "geotransform": ds.GetGeoTransform(can_return_null=True),
        "projection": ds.GetProjection() or None,
        "source_driver": ds.GetDriver().ShortName,
    }
    ds = None
    return rgb, meta


def _read_pillow(path: Path) -> tuple[np.ndarray, dict]:
    from PIL import Image

    with Image.open(path) as im:
        rgb = np.asarray(im.convert("RGB"), dtype=np.uint8)
    return rgb, {"reader": "pillow", "geotransform": None, "projection": None}


def read_image(path: str | Path) -> tuple[np.ndarray, dict]:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Нет файла: {path}")
    if path.suffix.lower() not in SUPPORTED_EXT:
        raise ValueError(f"Неподдерживаемый формат {path.suffix!r}: {path}")
    if HAS_GDAL:
        return _read_gdal(path)
    return _read_pillow(path)


def _write_gdal(path: Path, rgb: np.ndarray, meta: dict) -> None:
    ext = path.suffix.lower()
    driver_name = "GTiff" if ext in {".tif", ".tiff"} else "PNG"
    h, w, _ = rgb.shape
    if driver_name == "GTiff":
        ds = gdal.GetDriverByName("GTiff").Create(str(path), w, h, 3, gdal.GDT_Byte)
        if meta.get("geotransform"):
            ds.SetGeoTransform(meta["geotransform"])
        if meta.get("projection"):
            ds.SetProjection(meta["projection"])
        for i in range(3):
            ds.GetRasterBand(i + 1).WriteArray(rgb[:, :, i])
        ds.FlushCache()
        ds = None
    else:
        # PNG-драйвер GDAL не умеет Create — идём через MEM + CreateCopy
        mem = gdal.GetDriverByName("MEM").Create("", w, h, 3, gdal.GDT_Byte)
        for i in range(3):
            mem.GetRasterBand(i + 1).WriteArray(rgb[:, :, i])
        gdal.GetDriverByName("PNG").CreateCopy(str(path), mem)
        mem = None


def write_image(path: str | Path, rgb: np.ndarray, meta: dict | None = None) -> Path:
    path = Path(path)
    if rgb.ndim != 3 or rgb.shape[2] != 3 or rgb.dtype != np.uint8:
        raise ValueError(f"Ожидается uint8 (H, W, 3), получено {rgb.dtype} {rgb.shape}")
    path.parent.mkdir(parents=True, exist_ok=True)
    meta = meta or {}
    if HAS_GDAL:
        _write_gdal(path, rgb, meta)
    else:
        from PIL import Image

        if path.suffix.lower() in {".tif", ".tiff"} and meta.get("geotransform"):
            raise RuntimeError(
                "Гео-привязку без GDAL не сохранить — установите GDAL "
                "(sudo apt install python3-gdal)"
            )
        Image.fromarray(rgb).save(path)
    return path


def list_input_images(input_dir: str | Path, names: list[str] | None = None) -> list[Path]:
    input_dir = Path(input_dir)
    if names:
        paths = [input_dir / n for n in names]
        missing = [str(p) for p in paths if not p.is_file()]
        if missing:
            raise FileNotFoundError(f"Не найдены в {input_dir}: {missing}")
        return paths
    paths = sorted(
        p for p in input_dir.iterdir()
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXT
    )
    if not paths:
        raise FileNotFoundError(f"В {input_dir} нет изображений ({sorted(SUPPORTED_EXT)})")
    return paths


In [ ]:
%%writefile src/model_flux.py
"""Генераторы сезонных изображений за единым интерфейсом."""
from __future__ import annotations

import zlib
from dataclasses import dataclass

import numpy as np


@dataclass(frozen=True)
class GenRequest:
    source_name: str
    season: str
    ptype: str
    prompt: str
    negative_prompt: str | None
    seed: int


_SEASON_FX: dict[str, tuple[tuple[float, float, float], float]] = {
    "summer":      ((0.96, 1.10, 0.92), 0.00),
    "autumn":      ((1.18, 0.98, 0.72), 0.00),
    "winter":      ((1.00, 1.02, 1.10), 0.45),
    "deep_winter": ((1.00, 1.01, 1.06), 0.72),
}


class MockGenerator:

    name = "mock"
    model_id = "mock"

    def __init__(self, gcfg: dict | None = None):
        pass

    def generate(self, rgb: np.ndarray, req: GenRequest) -> np.ndarray:
        if req.season not in _SEASON_FX:
            raise ValueError(f"Неизвестный сезон {req.season!r}, есть: {list(_SEASON_FX)}")
        gains, white = _SEASON_FX[req.season]

        key = f"{req.ptype}|{req.season}|{req.source_name}|{req.seed}"
        h = zlib.crc32(key.encode("utf-8"))
        intensity = 0.85 + (h % 1000) / 1000.0 * 0.30

        img = rgb.astype(np.float64)
        for c in range(3):
            gain = 1.0 + (gains[c] - 1.0) * intensity
            img[:, :, c] *= gain
        w = min(white * intensity, 0.95)
        img = img * (1.0 - w) + 255.0 * w

        rng = np.random.default_rng(h)
        img += rng.normal(0.0, 2.0, size=img.shape)

        return np.clip(img, 0, 255).round().astype(np.uint8)


class FluxGenerator:
    """FLUX.1-Kontext-dev. По умолчанию bf16 (полная точность, весь пайплайн
    в VRAM). fp8 и gguf-q4 — опциональные режимы для GPU меньше 24/16 GB."""

    name = "flux"

    def __init__(self, gcfg: dict):
        import os
        os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF",
                              "expandable_segments:True")
        import torch
        from diffusers import FluxKontextPipeline

        self._torch = torch
        self.model_id = gcfg["model_id"]
        mode = gcfg.get("dtype", "bf16")
        if mode == "gguf-q4":
            dtype = torch.float16
            from diffusers import FluxTransformer2DModel, GGUFQuantizationConfig
            from transformers import BitsAndBytesConfig, T5EncoderModel

            transformer = FluxTransformer2DModel.from_single_file(
                gcfg["gguf_url"],
                quantization_config=GGUFQuantizationConfig(compute_dtype=dtype),
                config=gcfg["model_id"], subfolder="transformer",
                torch_dtype=dtype)
            transformer.to("cuda")  # T4-путь: держим в VRAM, RAM Colab всего 12.7GB
            text_encoder_2 = T5EncoderModel.from_pretrained(
                gcfg["model_id"], subfolder="text_encoder_2",
                quantization_config=BitsAndBytesConfig(load_in_8bit=True),
                device_map={"": 0})
            self.pipe = FluxKontextPipeline.from_pretrained(
                gcfg["model_id"], transformer=transformer,
                text_encoder_2=text_encoder_2, torch_dtype=dtype)
            self.pipe.text_encoder.to("cuda")
            self.pipe.vae.to("cuda")
            self.pipe.vae.enable_tiling()
        else:
            dtype = torch.bfloat16
            self.pipe = FluxKontextPipeline.from_pretrained(
                gcfg["model_id"], torch_dtype=dtype
            )
            if mode == "fp8":
                from optimum.quanto import freeze, qfloat8, quantize

                quantize(self.pipe.transformer, weights=qfloat8)
                freeze(self.pipe.transformer)
        if mode != "gguf-q4":
            offload = gcfg.get("cpu_offload", "model")
            if offload == "model":
                self.pipe.enable_model_cpu_offload()
            elif offload == "sequential":
                self.pipe.enable_sequential_cpu_offload()
            else:
                self.pipe.to("cuda")
        self.steps = int(gcfg["num_inference_steps"])
        self.guidance = float(gcfg["guidance_scale"])
        self.true_cfg = float(gcfg.get("true_cfg_scale", 1.0))
        self._dtype = dtype
        self._emb: dict[str, tuple] = {}

    def prepare_prompts(self, prompts: list[str]) -> None:
        """Прекомпьют эмбеддингов промптов, затем выгрузка текст-энкодеров."""
        import gc

        torch = self._torch
        with torch.inference_mode():
            for text in dict.fromkeys(prompts):
                emb, pooled, _ = self.pipe.encode_prompt(
                    prompt=text, prompt_2=None, device="cuda",
                    num_images_per_prompt=1)
                # каст к dtype пайплайна, иначе dtype-конфликт (mat1/mat2, VAE)
                self._emb[text] = (emb.to("cpu", self._dtype),
                                   pooled.to("cpu", self._dtype))
        self.pipe.text_encoder_2 = None
        self.pipe.text_encoder = None
        gc.collect()
        torch.cuda.empty_cache()

    def generate(self, rgb: np.ndarray, req: GenRequest) -> np.ndarray:
        from PIL import Image

        torch = self._torch
        emb, pooled = self._emb[req.prompt]
        kwargs = dict(
            image=Image.fromarray(rgb),
            num_inference_steps=self.steps,
            guidance_scale=self.guidance,
            generator=torch.Generator().manual_seed(req.seed),
            prompt_embeds=emb.to("cuda"),
            pooled_prompt_embeds=pooled.to("cuda"),
        )
        if req.negative_prompt:
            nemb, npooled = self._emb[req.negative_prompt]
            kwargs["negative_prompt_embeds"] = nemb.to("cuda")
            kwargs["negative_pooled_prompt_embeds"] = npooled.to("cuda")
            kwargs["true_cfg_scale"] = self.true_cfg
        out = self.pipe(**kwargs).images[0]
        return np.asarray(out.convert("RGB"), dtype=np.uint8)


class Flux2Generator:
    """Трансформер 64GB + энкодер Mistral 48GB не влезают в 95GB одновременно —
    работа через model_cpu_offload."""

    name = "flux2"
    model_id = "black-forest-labs/FLUX.2-dev"

    def __init__(self, gcfg: dict):
        import torch
        from diffusers import Flux2Pipeline

        self._torch = torch
        self.pipe = Flux2Pipeline.from_pretrained(
            self.model_id, torch_dtype=torch.bfloat16)
        self.pipe.enable_model_cpu_offload()
        self.steps = int(gcfg["num_inference_steps"])
        self.guidance = float(gcfg.get("flux2_guidance_scale", 2.5))

    def generate(self, rgb: np.ndarray, req: GenRequest) -> np.ndarray:
        from PIL import Image

        torch = self._torch
        out = self.pipe(
            prompt=req.prompt,
            image=[Image.fromarray(rgb)],
            num_inference_steps=self.steps,
            guidance_scale=self.guidance,
            generator=torch.Generator().manual_seed(req.seed),
        ).images[0]
        return np.asarray(out.convert("RGB"), dtype=np.uint8)


class SDXLImg2ImgGenerator:
    """CLIP режет промпт до 77 токенов."""

    name = "sdxl"
    model_id = "stabilityai/stable-diffusion-xl-base-1.0"

    def __init__(self, gcfg: dict):
        import torch
        from diffusers import StableDiffusionXLImg2ImgPipeline

        self._torch = torch
        self.pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
            self.model_id, torch_dtype=torch.float16, variant="fp16")
        self.pipe.to("cuda")
        self.pipe.vae.enable_tiling()
        self.strength = float(gcfg.get("sd_strength", 0.45))
        self.steps = int(gcfg["num_inference_steps"])
        self.guidance = float(gcfg.get("sd_guidance_scale", 7.5))

    def generate(self, rgb: np.ndarray, req: GenRequest) -> np.ndarray:
        from PIL import Image

        torch = self._torch
        out = self.pipe(
            prompt=req.prompt,
            negative_prompt=req.negative_prompt,
            image=Image.fromarray(rgb),
            strength=self.strength,
            num_inference_steps=self.steps,
            guidance_scale=self.guidance,
            generator=torch.Generator().manual_seed(req.seed),
        ).images[0]
        return np.asarray(out.convert("RGB"), dtype=np.uint8)


class InstructPix2PixGenerator:

    name = "ip2p"
    model_id = "timbrooks/instruct-pix2pix"

    def __init__(self, gcfg: dict):
        import torch
        from diffusers import StableDiffusionInstructPix2PixPipeline

        self._torch = torch
        self.pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
            self.model_id, torch_dtype=torch.float16, safety_checker=None)
        self.pipe.to("cuda")
        self.steps = int(gcfg["num_inference_steps"])
        self.guidance = float(gcfg.get("sd_guidance_scale", 7.5))
        self.image_guidance = float(gcfg.get("image_guidance_scale", 1.5))

    def generate(self, rgb: np.ndarray, req: GenRequest) -> np.ndarray:
        from PIL import Image

        torch = self._torch
        out = self.pipe(
            prompt=req.prompt,
            negative_prompt=req.negative_prompt,
            image=Image.fromarray(rgb),
            num_inference_steps=self.steps,
            guidance_scale=self.guidance,
            image_guidance_scale=self.image_guidance,
            generator=torch.Generator().manual_seed(req.seed),
        ).images[0]
        return np.asarray(out.convert("RGB"), dtype=np.uint8)


_BACKENDS = {
    "mock": MockGenerator,
    "flux": FluxGenerator,
    "flux2": Flux2Generator,
    "sdxl": SDXLImg2ImgGenerator,
    "ip2p": InstructPix2PixGenerator,
}


def get_generator(config: dict):
    gcfg = config["generation"]
    backend = gcfg.get("backend", "mock")
    if backend not in _BACKENDS:
        raise ValueError(f"Неизвестный backend {backend!r}, "
                         f"есть: {' | '.join(_BACKENDS)}")
    return _BACKENDS[backend](gcfg)


In [ ]:
%%writefile src/generate_images.py
"""Прогон матрицы генерации (исходник x сезон x тип промпта)."""
from __future__ import annotations

import argparse
import itertools
import json
import re
import shutil
import time
from datetime import datetime
from pathlib import Path

import numpy as np

import gdal_io
from model_flux import GenRequest, get_generator
from prompts import ROOT, build_prompt_matrix, load_config, load_prompts


def make_run_dir(base: Path) -> Path:
    run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
    for suffix in [""] + [f"-{i}" for i in range(1, 100)]:
        run_dir = base / f"{run_id}{suffix}"
        try:
            run_dir.mkdir(parents=True, exist_ok=False)
            return run_dir
        except FileExistsError:
            continue
    raise RuntimeError(f"Не удалось создать уникальную папку прогона в {base}")


def resize_to_resolution(rgb: np.ndarray, resolution: int) -> np.ndarray:
    from PIL import Image

    h, w = rgb.shape[:2]
    scale = resolution / max(h, w)
    new_w = max(16, round(w * scale / 16) * 16)
    new_h = max(16, round(h * scale / 16) * 16)
    if (new_h, new_w) == (h, w):
        return rgb
    im = Image.fromarray(rgb).resize((new_w, new_h), Image.LANCZOS)
    return np.asarray(im, dtype=np.uint8)


def sanitize(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_-]+", "_", name)


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--config", default=str(ROOT / "config.yaml"))
    ap.add_argument("--limit", type=int, default=0,
                    help="взять только первые N исходников (0 = все)")
    args = ap.parse_args()

    config = load_config(Path(args.config))
    prompts = load_prompts()
    matrix = build_prompt_matrix(config, prompts)
    gcfg = config["generation"]

    sources = gdal_io.list_input_images(
        ROOT / config["dataset"]["input_dir"],
        config["dataset"].get("images") or None,
    )
    if args.limit > 0:
        sources = sources[: args.limit]

    generator = get_generator(config)
    if hasattr(generator, "prepare_prompts"):
        texts = [t for item in matrix
                 for t in (item.prompt, item.negative_prompt) if t]
        generator.prepare_prompts(texts)
    run_dir = make_run_dir(ROOT / config["output"]["generated_dir"])

    shutil.copy2(args.config, run_dir / "run_config.yaml")
    shutil.copy2(ROOT / "prompts" / "prompts.yaml", run_dir / "prompts.yaml")

    variants = int(gcfg.get("num_variants", 1))
    total = len(sources) * len(matrix) * variants
    print(f"Backend: {generator.name} | исходников: {len(sources)} | "
          f"промптов: {len(matrix)} | вариантов: {variants} | "
          f"всего генераций: {total}")
    print(f"Папка прогона: {run_dir}")

    meta_path = run_dir / "metadata.jsonl"
    done = 0
    t_run = time.perf_counter()
    with meta_path.open("w", encoding="utf-8") as meta_f:
        for src_path in sources:
            rgb, io_meta = gdal_io.read_image(src_path)
            rgb = resize_to_resolution(rgb, int(gcfg["resolution"]))
            for item, variant in itertools.product(matrix, range(variants)):
                seed = int(gcfg["seed"]) + variant
                req = GenRequest(
                    source_name=src_path.name,
                    season=item.season,
                    ptype=item.ptype,
                    prompt=item.prompt,
                    negative_prompt=item.negative_prompt,
                    seed=seed,
                )
                t0 = time.perf_counter()
                out_rgb = generator.generate(rgb, req)
                gen_time = time.perf_counter() - t0

                out_name = (f"{sanitize(src_path.stem)}__{item.season}__"
                            f"{item.ptype}__v{variant}.png")
                gdal_io.write_image(run_dir / out_name, out_rgb, io_meta)

                record = {
                    "run_id": run_dir.name,
                    "source": src_path.name,
                    "season": item.season,
                    "ptype": item.ptype,
                    "variant": variant,
                    "prompt": item.prompt,
                    "negative_prompt": item.negative_prompt,
                    "seed": seed,
                    "steps": int(gcfg["num_inference_steps"]),
                    "guidance_scale": float(gcfg["guidance_scale"]),
                    "true_cfg_scale": (float(gcfg["true_cfg_scale"])
                                       if item.negative_prompt else None),
                    "backend": generator.name,
                    "model_id": generator.model_id,
                    "dtype": gcfg["dtype"],
                    "resolution": list(out_rgb.shape[:2]),
                    "out_file": out_name,
                    "gen_time_sec": round(gen_time, 3),
                    "timestamp": datetime.now().isoformat(timespec="seconds"),
                }
                meta_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                done += 1
                print(f"  [{done}/{total}] {out_name} ({gen_time:.2f}s)")

    print(f"Готово: {done} изображений за {time.perf_counter() - t_run:.1f}s -> {run_dir}")


if __name__ == "__main__":
    main()


In [ ]:
!pip install -q -U diffusers transformers accelerate safetensors sentencepiece optimum-quanto gguf bitsandbytes
import diffusers
print('diffusers', diffusers.__version__)

In [ ]:
HF_TOKEN = ''  # после практики токен отозвать
from huggingface_hub import login
login(token=HF_TOKEN)
print('HF login OK')

In [ ]:
# Данные: те же 10 снимков val/A LEVIR-CD, что и локально
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download
zp = hf_hub_download('satellite-image-deep-learning/LEVIR-CD',
                     'val.zip', repo_type='dataset')
with zipfile.ZipFile(zp) as z:
    a = sorted([n for n in z.namelist()
                if n.startswith('A/') and n.endswith('.png')],
               key=lambda n: int(''.join(c for c in n if c.isdigit())))
    picks = a[::max(1, len(a) // 10)][:10]
    for name in picks:
        Path('data/input', 'levir_' + Path(name).name).write_bytes(z.read(name))
print('исходников:', len(list(Path('data/input').glob('*.png'))))

In [ ]:
import yaml
cfg = yaml.safe_load(open('config.yaml', encoding='utf-8'))
cfg['generation'].update(backend='flux', dtype=DTYPE, cpu_offload=OFFLOAD)
yaml.safe_dump(cfg, open('config_colab.yaml', 'w', encoding='utf-8'),
               allow_unicode=True, sort_keys=False)
smoke = {**cfg, 'seasons': ['winter'], 'prompt_types': ['contextual']}
yaml.safe_dump(smoke, open('config_smoke.yaml', 'w', encoding='utf-8'),
               allow_unicode=True, sort_keys=False)
print('конфиги готовы')

In [ ]:
# SMOKE: 1 исходник x 1 промпт (первый запуск скачает модель ~24GB)
!python src/generate_images.py --config config_smoke.yaml --limit 1

In [ ]:
import pathlib
from IPython.display import display
from PIL import Image
run = sorted(pathlib.Path('outputs/generated').iterdir())[-1]
out = next(run.glob('*.png'))
src = sorted(pathlib.Path('data/input').glob('*.png'))[0]
print(out.name)
display(Image.open(src).resize((384, 384)),
        Image.open(out).resize((384, 384)))

Если smoke ок — полный прогон (A100: ~1-1.5 ч на 240 генераций).

In [ ]:
!python src/generate_images.py --config config_colab.yaml

In [ ]:
import pathlib, shutil
from google.colab import files
run = sorted(pathlib.Path('outputs/generated').iterdir())[-1]
arc = shutil.make_archive(f'flux_results_{run.name}', 'zip',
                          run.parent, run.name)
print(arc)
files.download(arc)